# CredAIR — Análise de Risco de Crédito

**Objetivo:** usar o histórico de propostas de crédito da CredAIR para entender os fatores ligados à inadimplência e construir um modelo de **Regressão Logística** que ajude a identificar propostas de maior risco.

> Os dados são **sintéticos** e foram criados para fins educacionais.

### Roteiro do notebook

| Seção | Conteúdo | Camada de dados |
|---|---|---|
| 1 | Entendimento do problema | 🥉 bronze |
| 2 | Diagnóstico de qualidade dos dados | 🥉 bronze |
| 3 | Limpeza e preparação | 🥉 → 🥈 prata |
| 4 | Análise exploratória | 🥈 prata |
| 5 | Engenharia de atributos | 🥈 → 🥇 ouro |
| 6 | Preparação para modelagem | 🥇 ouro |
| 7 | Modelagem | 🥇 ouro |
| 8 | Avaliação e interpretação | — |
| 9 | Conclusão | — |

Ao longo do notebook, cada decisão importante aparece num bloco **🧭 Decisão**, com o que foi feito, por quê e qual alternativa foi descartada.

## 0. Configuração do ambiente

Importamos as bibliotecas e fixamos uma **semente aleatória** (`RANDOM_STATE`). Com ela, a divisão treino/teste e o treinamento produzem sempre os mesmos resultados, o que torna a análise **reprodutível**.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42

# Raiz do projeto: funciona tanto no VS Code (raiz = pasta do projeto)
# quanto no Jupyter aberto dentro de notebooks/
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BRONZE = RAIZ / "data" / "bronze" / "dataset.csv"
PRATA = RAIZ / "data" / "silver" / "credito_limpo.parquet"
OURO = RAIZ / "data" / "gold" / "credito_modelagem.parquet"
FIGURAS = RAIZ / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

# Estilo dos gráficos: fundo limpo, grade discreta e duas cores fixas para as classes
AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#898781"
CORES_CLASSE = {0: AZUL, 1: LARANJA}
sns.set_theme(style="whitegrid", rc={"grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7"})
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)


def salvar(fig, nome):
    # Salva a figura em reports/figures para usar na apresentação
    fig.savefig(FIGURAS / f"{nome}.png", dpi=120, bbox_inches="tight")

## 1. Entendimento do problema

### 1.1 Carregando os dados brutos (camada bronze)

Lemos o arquivo **exatamente como foi recebido**. Nenhuma alteração é feita nesta etapa.

In [2]:
bronze = pd.read_csv(BRONZE)
print(f"Linhas: {bronze.shape[0]:,} | Colunas: {bronze.shape[1]}")
bronze.head()

Linhas: 5,000 | Colunas: 11


,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
0,52,MS,Ensino Superior,"2,911.820",14,690,"5,934.190",60,Viagem,1,0
1,49,RR,Ensino Médio,"3,176.430",5,676,"6,791.380",24,Outros,2,0
2,48,AL,Ensino Superior,"3,429.400",23,558,"14,666.780",24,Outros,5,1
3,59,GO,Ensino Fundamental,"10,096.700",2,739,"19,053.240",24,Reforma,0,0
4,37,RJ,Ensino Fundamental,"3,033.250",24,797,"6,171.310",6,Educação,0,1


### 1.2 Dicionário de dados

| Coluna | Tipo | Papel | Descrição |
|---|---|---|---|
| `idade` | numérica | explicativa | Idade do solicitante na proposta |
| `estado` | categórica | explicativa | UF de residência |
| `escolaridade` | categórica **ordinal** | explicativa | Nível de escolaridade declarado |
| `renda_mensal` | numérica | explicativa | Renda mensal declarada (R$) |
| `tempo_emprego_anos` | numérica | explicativa | Anos no emprego atual |
| `score_credito` | numérica | explicativa | Pontuação de crédito do solicitante |
| `valor_solicitado` | numérica | explicativa | Valor do empréstimo (R$) |
| `prazo_meses` | numérica discreta | explicativa | Prazo de pagamento (meses) |
| `finalidade` | categórica | explicativa | Finalidade do empréstimo |
| `dividas_ativas` | numérica discreta | explicativa | Quantidade de dívidas em aberto |
| **`inadimplente`** | **binária (0/1)** | **resposta** | **1 = a proposta resultou em inadimplência** |

> **Divergência no enunciado:** o texto cita o arquivo `credito.csv`, mas o arquivo fornecido se chama `dataset.csv`. Mantivemos o nome original para preservar a rastreabilidade do dado bruto.

### 1.3 Variável resposta e tipo de problema

- **Variável resposta:** `inadimplente`. Indica se a proposta, **depois de acompanhada ao longo do tempo**, resultou em falta de pagamento (`1`) ou não (`0`).
- **Por que é Machine Learning supervisionado:** temos exemplos históricos **com a resposta conhecida**. O modelo aprende a relação entre as características da proposta e o resultado observado.
- **Por que é classificação, e não regressão:** a resposta é uma **categoria** (inadimplente ou não), e não uma quantidade contínua. Mais precisamente, é uma **classificação binária**.
- **O que o modelo vai entregar:** a Regressão Logística estima a **probabilidade** de inadimplência, um número entre 0 e 1. Para virar uma decisão (aprovar, recusar ou analisar), essa probabilidade é comparada com um **limiar**, que vamos discutir na seção 8.

### 1.4 O custo dos erros

No crédito, os dois tipos de erro **não custam o mesmo**:

| Erro | O que acontece | Custo para a CredAIR |
|---|---|---|
| **Falso negativo:** o modelo diz "bom pagador", mas a pessoa fica inadimplente | Empréstimo aprovado e não pago | **Perda do capital emprestado** (alto) |
| **Falso positivo:** o modelo diz "risco", mas a pessoa pagaria | Proposta recusada sem necessidade | Perda dos juros que seriam recebidos e do cliente (moderado) |

Por isso, na avaliação, **não basta olhar a acurácia**. Vamos dar atenção especial ao **recall da classe inadimplente**, ou seja, a fração dos maus pagadores que o modelo consegue identificar.

In [3]:
contagem = bronze["inadimplente"].value_counts().sort_index()
pd.DataFrame({
    "quantidade": contagem,
    "proporção": (contagem / contagem.sum()).map("{:.1%}".format),
}).rename(index={0: "0 - adimplente", 1: "1 - inadimplente"})

,quantidade,proporção
inadimplente,,
0 - adimplente,3280,65.6%
1 - inadimplente,1720,34.4%


**Primeira leitura:** cerca de **34%** das propostas resultaram em inadimplência. As classes são **moderadamente desbalanceadas**. Um "modelo" que dissesse sempre "adimplente" acertaria cerca de 66% das vezes sem aprender nada. Por isso a acurácia sozinha engana, e vamos comparar o modelo com essa referência na seção 7.